# Chapter 10: Weak Supervision

This notebook accompanies **Chapter 10** of the lecture notes.

> Today there is *some* supervision — but it is partial, noisy, and conflicting. A heuristic rule with high precision on a few cases. A crowd annotator who flips occasionally. A pretrained classifier that is confidently wrong outside its training distribution. The chapter abstracts all three into a single object — the *labelling function* — and asks how to train a model when nobody fully trusts any single source.

**Agenda**

🛠️ · 🧮 · 🎓 · ⚖️ · 🏁

**Take it from here:** 🔍 · 🪜


In [ ]:
import numpy as np
import sys; sys.path.insert(0, '../..')
from plot_style import *
from sklearn.linear_model import LogisticRegression
from checks import (
    check_lf_coverage, check_lf_empirical_accuracy,
    check_majority_vote, check_dawid_skene_em,
    check_soft_cross_entropy, check_pick_most_uncertain,
    ABSTAIN,
)
from viz_helpers import (
    load_binary_digits, apply_lfs,
    lf_top_loop, lf_noisy_crowd, lf_pretrained, make_judge,
    train_logistic_on_soft, predict_proba,
    run_active_learning, run_random_baseline,
    plot_active_learning_curve, show_lf_examples,
)

RNG = np.random.default_rng(0)


## 🛠️ Labelling Functions

A *labelling function* (LF) is a Python function that takes an input and returns either a label or `ABSTAIN`. We work on a binary task: classify 8×8 sklearn digits as **4 vs 9** (label `0` = 4, `1` = 9). The training pool is unlabelled; a 20-example *dev set* is the only labelled data we have, used to read off each LF's accuracy.

We define three LFs, one of each type the chapter calls out:
- **`lf_top_loop`** — a hand-crafted heuristic: a 9 has a closed loop at the top, a 4 has an open top.
- **`lf_noisy_crowd`** — a simulated crowd annotator: knows the truth most of the time, flips with probability 0.2, abstains on 30% of the data.
- **`lf_pretrained`** — a small pretrained classifier (`LogisticRegression` on 10 labelled examples) that abstains when its top-class probability is below a threshold.


In [ ]:
X, y, X_test, y_test, X_dev, y_dev = load_binary_digits(seed=0)
print(f'Training pool: {X.shape}  (no labels)   Test: {X_test.shape}   Dev: {X_dev.shape}')

# Apply the lead heuristic to the pool and see where it fires.
out_lead = lf_top_loop(X)
show_lf_examples(X, y, out_lead, 'lf_top_loop', n_each=4)


**Coverage** = fraction of inputs the LF didn't abstain on. **Accuracy** (on the dev set) = of the times it fired, how often it was right. The two together describe an LF's precision–coverage trade-off.

Implement `lf_coverage(L)` and `lf_empirical_accuracy(L_dev, y_dev)`. `L` has shape `(n_samples, n_lfs)` with values in `{0, 1, ABSTAIN}`.


In [ ]:
def lf_coverage(L):
    """Fraction of non-abstain entries per LF. Returns shape (n_lfs,)."""
    # YOUR CODE HERE
    pass


check_lf_coverage(lf_coverage)


In [ ]:
def lf_empirical_accuracy(L_dev, y_dev):
    """Per-LF accuracy on the dev set, conditioned on non-abstain rows."""
    # YOUR CODE HERE
    pass


check_lf_empirical_accuracy(lf_empirical_accuracy)


### The 3-LF stack

We build the label matrix `L` once. The crowd LF needs ground truth (it's a simulated annotator), so we apply it per split with the corresponding labels and column-stack with the pure-input LFs. Every LF ends up in the same `(n_samples, n_lfs)` matrix; downstream nothing knows where each column came from.


In [ ]:
# Tiny labelled subset for the pretrained LF (5 of each class)
_idx_pre = np.concatenate([np.where(y_dev == 0)[0][:5], np.where(y_dev == 1)[0][:5]])
_X_pre, _y_pre = X_dev[_idx_pre], y_dev[_idx_pre]

LF_NAMES = ['top_loop', 'noisy_crowd', 'pretrained']

def build_L(Xs, ys, seed):
    return np.column_stack([
        lf_top_loop(Xs),
        lf_noisy_crowd(Xs, ys, flip_rate=0.20, abstain_rate=0.30, seed=seed),
        lf_pretrained(Xs, _X_pre, _y_pre),
    ])

L      = build_L(X,     y,     seed=0)
L_dev  = build_L(X_dev, y_dev, seed=1)
print(f'L: {L.shape}, values in {{0, 1, -1}}')

if lf_coverage(L) is not None and lf_empirical_accuracy(L_dev, y_dev) is not None:
    cov = lf_coverage(L)
    acc = lf_empirical_accuracy(L_dev, y_dev)
    print(f'\n{"LF":<14}  coverage  accuracy')
    for n, c, a in zip(LF_NAMES, cov, acc):
        a_str = f'{a:.2f}' if not np.isnan(a) else '   nan'
        print(f'{n:<14}    {c:.2f}      {a_str}')


## 🧮 Label Model

Three LFs, conflicting votes, no ground truth on the training pool. We need one label per example. The naive aggregation is **majority vote**: count non-abstain votes, pick the winner. The principled aggregation is **Dawid–Skene with EM**: estimate each LF's accuracy from agreement structure alone, then weight votes by inferred accuracy.

> The label model has no labelled data. What lets it tell a good LF from a bad one?

<details><summary>Thought</summary>

Agreement is the signal. If two LFs agree on 80% of the rows where both fire, the model that calls them both 50% accurate predicts 50% agreement; calling them both 80% predicts 0.8² + 0.2² = 68%; 90% predicts 82%. The likelihood of the observed agreement pattern is highest at the right accuracies, and EM climbs that likelihood. The trivial all-0.5 fixed point is a saddle — a small initial bias toward "LFs are better than random" lets EM find the data-explaining solution.
</details>


In [ ]:
def majority_vote(L):
    """Per-row majority of non-abstain votes; ABSTAIN on ties."""
    # YOUR CODE HERE
    pass


check_majority_vote(majority_vote)


In [ ]:
def dawid_skene_em(L, n_iters):
    """One-coin Dawid-Skene model fitted by EM.

    Returns (alphas, soft_pos):
      alphas    : (n_lfs,) each LF's inferred accuracy
      soft_pos  : (n_samples,) P(y = 1 | L) for every example.
    """
    # E-step: for each example, log p(y=1 | L) - log p(y=0 | L).
    #   For LF i with current accuracy a_i, a vote of 1 contributes log(a_i / (1 - a_i)),
    #   a vote of 0 contributes log((1 - a_i) / a_i), and an abstain contributes 0.
    # Sigmoid the sum to get soft_pos.
    # M-step: each a_i is the agreement between its non-abstain votes and soft_pos
    #   (clip to [0.5 + 1e-3, 1 - 1e-3] so EM cannot collapse to 0.5 or invert polarity).
    # Initialise alphas at 0.7 and run for n_iters loops.
    # YOUR CODE HERE
    pass


check_dawid_skene_em(dawid_skene_em)


In [ ]:
# Aggregate the label matrix and compare on the held-out test set.
out_dse = dawid_skene_em(L, 30)
out_mv  = majority_vote(L)

if out_dse is not None and out_mv is not None:
    alphas, soft_pos = out_dse
    print(f'{"LF":<14}  recovered accuracy   dev accuracy')
    acc_emp = lf_empirical_accuracy(L_dev, y_dev)
    for n, a, e in zip(LF_NAMES, alphas, acc_emp):
        e_str = f'{e:.2f}' if not np.isnan(e) else '   nan'
        print(f'{n:<14}     {a:.2f}              {e_str}')

    # Test-set accuracy
    L_test = build_L(X_test, y_test, seed=2)
    mv_test = majority_vote(L_test)
    keep = mv_test != ABSTAIN
    print(f'\nMajority vote on test : {(mv_test[keep] == y_test[keep]).mean():.2%}  '
          f'({keep.sum()}/{len(y_test)} covered)')

    _, soft_test = dawid_skene_em(L_test, 30)
    dse_acc = float(((soft_test > 0.5).astype(int) == y_test).mean())
    print(f'DS-EM argmax on test  : {dse_acc:.2%}  (every example labelled)')


## 🎓 End Model

The label model gave us a soft posterior per training example. Train a logistic regression on the *soft* labels — same shape as cross-entropy, but the target is a probability rather than a one-hot. This is the chapter's *distillation* step: the label model is the teacher, the end model is the student, the soft posterior is the temperature-softened target. The end model then generalises to inputs no LF fired on, because it operates on the input features rather than on LF outputs.


In [ ]:
def soft_cross_entropy(soft_labels, model_probs):
    """Binary cross-entropy with soft targets in [0, 1]."""
    # YOUR CODE HERE
    pass


check_soft_cross_entropy(soft_cross_entropy)


In [ ]:
out_dse = dawid_skene_em(L, 30)
if out_dse is not None and soft_cross_entropy(np.array([0.5]), np.array([0.5])) is not None:
    _, soft_pos_train = out_dse

    w_end, b_end = train_logistic_on_soft(X, soft_pos_train, soft_cross_entropy,
                                           n_iters=80, seed=0)
    end_acc = float(((predict_proba(X_test, w_end, b_end) > 0.5).astype(int) == y_test).mean())
    print(f'End model (LR on soft labels)        : {end_acc:.2%}')
    print(f'LR on FULL ground truth (oracle)     : '
          f'{LogisticRegression(max_iter=200).fit(X, y).score(X_test, y_test):.2%}')

    # Verifier-generator gap: examples on which the FEWEST LFs fired.
    n_votes = (L != ABSTAIN).sum(axis=1)
    sparse_idx = np.argsort(n_votes)[:20]
    sparse_acc = float(((predict_proba(X[sparse_idx], w_end, b_end) > 0.5).astype(int)
                        == y[sparse_idx]).mean())
    print(f'\nEnd-model accuracy on the 20 sparsest-coverage rows: {sparse_acc:.2%}')
    print('The end model labels rows the LFs barely covered — '
          'verifier-generator gap in action.')


## ⚖️ LLM-as-Judge

A *judge* is a labelling function whose decision is delegated to a stronger model — typically an LLM, sometimes a separately-trained classifier. From the aggregation pipeline's perspective it is just another LF: same `(input → vote ∪ abstain)` interface, slots into the same `L` matrix. Pyodide can't host an LLM, so the judge here is a 3-layer MLP trained on the full dev set — same role at toy scale.


In [ ]:
# Build the judge and add it as a fourth column of the label matrix.
judge_lf = make_judge(X_dev, y_dev)

L_j      = np.column_stack([L,     judge_lf(X)     ])
L_test_j = np.column_stack([L_test, judge_lf(X_test)])
LF_NAMES_J = LF_NAMES + ['judge']

if dawid_skene_em(L_j, 5) is not None:
    alphas_j, soft_j = dawid_skene_em(L_j, 30)
    print(f'{"LF":<14}  recovered accuracy')
    for n, a in zip(LF_NAMES_J, alphas_j):
        print(f'{n:<14}     {a:.2f}')

    w_j, b_j = train_logistic_on_soft(X, soft_j, soft_cross_entropy, n_iters=80, seed=0)
    end_acc_j = float(((predict_proba(X_test, w_j, b_j) > 0.5).astype(int) == y_test).mean())
    print(f'\nEnd-model accuracy with judge: {end_acc_j:.2%}')


### 🏁 Recap

- 🛠️ Three different supervision sources — heuristic rule, noisy crowd, pretrained classifier — collapsed to a single object: a function returning a vote or `ABSTAIN`.
- 🧮 Dawid–Skene EM recovered per-LF accuracies from agreement structure alone, no ground truth needed; aggregated soft posterior per example.
- 🎓 The end model trained on soft labels generalised to inputs no LF fired on — the verifier-generator gap.
- ⚖️ The judge slotted into the same pipeline as a fourth LF; the label model re-aggregated and the end model retrained.

The next chapter picks up the loop in a different mode: the model labels its own data and trains on its own predictions, with calibration and confirmation bias as the central failure modes.


## Take It from Here, Next Steps

### 🔍 Active learning — the opposite axis

Weak supervision sacrifices per-label *quality* for coverage; active learning sacrifices coverage for per-label *quality*. The two are complementary: cheap noisy labels scale the training set, a small budget of clean labels directs the model where it most needs them.

The simplest criterion is *uncertainty sampling*: ask the oracle about the example whose predicted `P(y=1)` is closest to 0.5.


In [ ]:
def pick_most_uncertain(model_probs):
    """Index of the entry whose predicted probability is closest to 0.5."""
    # YOUR CODE HERE
    pass


check_pick_most_uncertain(pick_most_uncertain)


In [ ]:
out_dse = dawid_skene_em(L, 30)
if (out_dse is not None and soft_cross_entropy(np.array([0.5]), np.array([0.5])) is not None
    and pick_most_uncertain(np.array([0.4, 0.5, 0.6])) is not None):
    _, soft_pos_train = out_dse
    n_q = 12

    _, accs_unc = run_active_learning(
        X, y, X_test, y_test, soft_pos_train,
        soft_cross_entropy, pick_most_uncertain, n_queries=n_q, seed=0)
    _, accs_rnd = run_random_baseline(
        X, y, X_test, y_test, soft_pos_train,
        soft_cross_entropy, n_queries=n_q, seed=0)
    plot_active_learning_curve(list(range(n_q + 1)), accs_unc, accs_rnd)


### 🪜 Process reward models

The judge in §⚖️ scored a final output. A *process reward model* generalises the construction by scoring *intermediate steps* — given a partial reasoning trajectory or tool-use trace, the PRM emits a per-step quality estimate. The dense per-step signal is what converts a sparse terminal reward into something a policy can credit-assign through. Same machinery as §🎓 — soft cross-entropy on a probabilistic target — applied to a richer state.
